# Task 4: Statistical Modeling

In this notebook, we build models to predict `TotalClaims` based on the available features. We evaluate Linear Regression, Random Forest, and XGBoost, and implement SHAP for interpretability.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import shap
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

### Data Preprocessing

In [ ]:
df = pd.read_csv('../data/raw/insurance_data.csv')

X = df.drop(columns=['TotalClaims'])
y = df['TotalClaims']

numerical_cols = ['TotalPremium']
categorical_cols = ['ZipCode', 'Province', 'Gender', 'VehicleMake']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Model Training and Evaluation

In [ ]:
def evaluate_model(name, model):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"--- {name} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R2: {r2:.4f}\n")
    return pipeline

lr_pipeline = evaluate_model('Linear Regression', LinearRegression())
rf_pipeline = evaluate_model('Random Forest', RandomForestRegressor(random_state=42))
xgb_pipeline = evaluate_model('XGBoost', XGBRegressor(random_state=42))

### Interpretability with SHAP
We will explain the XGBoost model predictions using SHAP values.

In [ ]:
# Preprocess the training data to feed into SHAP explainer
X_train_processed = preprocessor.fit_transform(X_train)
feature_names = (numerical_cols + 
                 list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)))

X_train_proc_df = pd.DataFrame(X_train_processed, columns=feature_names)

xgb_model = xgb_pipeline.named_steps['model']
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train_proc_df)

shap.summary_plot(shap_values, X_train_proc_df, plot_type="bar")